# J10 — Transport de solutés et modélisation inverse

**Physique et hydrodynamique des sols** — S. J. Gumiere

*Atelier du Jour 10 : solutions analytiques de l'ADE, courbes de percée HYDRUS-1D, soluté réactif, estimation inverse de paramètres*

> Version **étudiant** : complétez les cellules marquées `# À COMPLÉTER`.

## Mise en place

Colonne de référence (projets `J10_*`) : loam sableux, régime permanent $q = 10$ cm/j vers le bas, $\theta = 0{,}3293$,
$v = q/\theta = 30{,}4$ cm/j, $L = 60$ cm, pulse $c_0 = 1$ pendant $t_0 = 0{,}5$ j (masse injectée $q c_0 t_0 = 5$), $D_0 = 1$ cm²/j,
$\tau = \theta^{7/3}/\theta_s^2$ (Millington–Quirk), $D = \lambda v + D_0\tau$. Convention HYDRUS : flux positif vers le haut
(`cvBot` < 0 = sortie par le bas) ; dans les solutions analytiques, $x$ est la distance parcourue vers le bas.

In [ ]:
import sys, time
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import special, optimize, linalg, integrate
from hydrus_io import read_solute, read_obs_node, read_nod_inf, read_tlevel

plt.rcParams.update({"figure.figsize": (7.5, 4), "axes.grid": True, "grid.alpha": 0.3})
HYD = "../../hydrus"
q, theta, theta_s = 10.0, 0.3293, 0.41
v = q / theta
tau = theta ** (7 / 3) / theta_s ** 2
De = 1.0 * tau
L, t0 = 60.0, 0.5
print(f"v = {v:.2f} cm/j ; tau = {tau:.3f} ; De = {De:.3f} cm²/j")

## Exercice 1 — Solutions analytiques de l'équation d'advection–dispersion  *(≈ 15 min)*

1. Implémenter `ade_step_flux(x, t, v, D, R=1, mu=0)` : concentration de flux $c_f/c_0$ pour une injection continue avec condition de flux
   à l'entrée (forme d'Ogata–Banks généralisée) : avec $v' = v/R$, $D' = D/R$, $u = v'\sqrt{1 + 4\mu D'/v'^2}$,
   $$\frac{c}{c_0} = \tfrac12 e^{(v'-u)x/2D'}\,\mathrm{erfc}\!\left(\frac{x - ut}{2\sqrt{D't}}\right) + \tfrac12 e^{(v'+u)x/2D'}\,\mathrm{erfc}\!\left(\frac{x + ut}{2\sqrt{D't}}\right).$$
   Implémenter aussi `ade_step_resident(x, t, v, D, R=1)` (concentration résidente, van Genuchten & Alves 1982, cas A2, sans dégradation) :
   $$\frac{c}{c_0} = \tfrac12\,\mathrm{erfc}\!\left(\frac{x - v't}{2\sqrt{D't}}\right) + \sqrt{\frac{v'^2 t}{\pi D'}}\,e^{-\frac{(x - v't)^2}{4D't}}
   - \tfrac12\left(1 + \frac{v'x}{D'} + \frac{v'^2 t}{D'}\right) e^{v'x/D'}\,\mathrm{erfc}\!\left(\frac{x + v't}{2\sqrt{D't}}\right).$$
2. Écrire `ade_pulse(x, t, v, D, t0, R=1, mu=0, kind="flux")` par superposition de deux échelons.
3. Pour $\lambda$ = 0,5, 2 et 10 cm : tracer les profils résidents $c(z)$ à $t$ = 0,25, 0,5, 1, 1,5, 2 j et les BTC (flux) à 60 cm ;
   vérifier la masse $\int_0^\infty q\,c_f\,dt = 5$ (`np.trapezoid`) et calculer $Pe = vL/D$ ; commenter la position et la hauteur du pic.
4. Vérifier que $c_f$ et $c_r$ diffèrent d'autant plus que $Pe$ est petit (écart maximal pour $\lambda$ = 0,5, 2, 10 cm).

In [ ]:
def ade_step_flux(x, t, v, D, R=1.0, mu=0.0):
    t = np.asarray(t, float); x = np.asarray(x, float)
    vr, Dr = v / R, D / R
    u = vr * np.sqrt(1 + 4 * mu * Dr / vr ** 2)
    # À COMPLÉTER (attention à t = 0 : utiliser np.errstate et renvoyer 0)
    pass

def ade_step_resident(x, t, v, D, R=1.0):
    # À COMPLÉTER
    pass

def ade_pulse(x, t, v, D, t0, R=1.0, mu=0.0, kind="flux"):
    f = ade_step_flux if kind == "flux" else (lambda x, t, v, D, R=1.0, mu=0.0: ade_step_resident(x, t, v, D, R))
    # À COMPLÉTER : superposition f(x, t) - f(x, max(t - t0, 0))
    pass

# 3. profils et BTC pour lambda = 0.5, 2, 10 ; masse ; Pe
# 4. c_f vs c_r

## Exercice 2 — Courbes de percée HYDRUS-1D et méthode des moments  *(≈ 15 min)*

Projets `J10_traceur_disp0.5cm`, `J10_traceur_disp2cm`, `J10_traceur_disp10cm` (mêmes conditions, $\lambda$ = 0,5 / 2 / 10 cm).

1. Lire `solute1.out` (`read_solute` : `cvBot`, `Sum(cvBot)`, `Sum(cvTop)`) et `OBS_NODE.OUT` (`read_obs_node`, variable `Conc`, nœuds 21, 41, 81, 121
   = 10, 20, 40, 60 cm). Tracer les BTC HYDRUS ($c_f = -$`cvBot`$/q$ à la sortie ; $c_r$ aux nœuds) avec les solutions analytiques correspondantes
   (flux à la sortie, résidente aux nœuds) ; RMSE par projet.
2. Bilan de masse : `Sum(cvTop)` (injectée) vs `Sum(cvBot)` à 6 j.
3. Moments temporels de $c_f(60, t)$ : $M_0 = q\int c_f\,dt$, $\bar t$, $\sigma_t^2$ ; en déduire $v = L/(\bar t - t_0/2)$ et
   $D = (\sigma_t^2 - t_0^2/12)\,v^3/(2L)$ ; comparer aux valeurs imposées ; expliquer l'écart pour $\lambda$ = 10 cm.
4. Profils `NOD_INF.OUT` (colonne `Conc(1..NS)`) à 0,5, 1, 1,5 j pour $\lambda$ = 2 cm vs solution résidente.

In [ ]:
projets = {0.5: "J10_traceur_disp0.5cm", 2.0: "J10_traceur_disp2cm", 10.0: "J10_traceur_disp10cm"}
noeuds = {21: 10, 41: 20, 81: 40, 121: 60}
# 1. BTC HYDRUS vs analytique : so = read_solute(f"{HYD}/{nom}") ; ob = read_obs_node(f"{HYD}/{nom}") ; c_f = -so["cvBot"] / q
# À COMPLÉTER

# 2. bilan de masse : so["Sum(cvTop)"], so["Sum(cvBot)"] à t = 6 j
# À COMPLÉTER

# 3. moments -> v, D (np.trapezoid)
# À COMPLÉTER

# 4. profils NOD_INF (colonne "Conc(1..NS)")
# À COMPLÉTER

## Exercice 3 — Soluté réactif : retard, dégradation et ajustement de ($K_d$, $\mu$)  *(≈ 15 min)*

`J10_nitrate_reactif` : mêmes conditions que `J10_traceur_disp2cm` avec sorption linéaire $K_d = 0{,}2$ cm³/g ($\rho_b = 1{,}5$ g/cm³) et
dégradation du premier ordre $\mu = 0{,}05$ /j (phases liquide et adsorbée).

1. Tracer les BTC de sortie du traceur et du soluté réactif (HYDRUS) et les solutions analytiques ($R = 1 + \rho_b K_d/\theta$, $\mu$).
   Temps du pic et temps moyen ; rapport au traceur ; comparer à $R$.
2. Masse récupérée à 6 j (`Sum(cvBot)`) vs masse injectée ; comparer à $M e^{-\mu \bar t}$ et à l'intégrale de la solution analytique
   jusqu'à $t \to \infty$ (`scipy.integrate.quad`).
3. Ajuster ($K_d$, $\mu$) par `curve_fit` sur la BTC HYDRUS avec `ade_pulse` ($v$, $D$ connus) depuis (0,1 ; 0,02) ; écart-types,
   corrélation $K_d$–$\mu$ ; tracer l'ajustement. Que se passe-t-il si l'on n'utilise que la partie montante de la BTC ($t < 3{,}5$ j) ?

In [ ]:
rho_b, Kd, mu = 1.5, 0.2, 0.05
R = 1 + rho_b * Kd / theta
D = 2.0 * v + De
so_t = read_solute(f"{HYD}/J10_traceur_disp2cm"); so_n = read_solute(f"{HYD}/J10_nitrate_reactif")
# 1. BTC et temps caractéristiques
# 2. masses
# 3. curve_fit sur (Kd, mu)
def btc_reactif(t, Kd_, mu_):
    # À COMPLÉTER : ade_pulse(60, t, v, D, t0, R=1 + rho_b*Kd_/theta, mu=mu_)
    pass

## Exercice 4 — Modélisation inverse : estimer ($\alpha$, $n$, $K_s$) avec un solveur de Richards  *(≈ 20 min)*

`data/J10_inverse_donnees.csv` : infiltration cumulée $I(t)$ ($\sigma_I$ = 0,2 cm) et charges de pression à 10, 30 et 50 cm ($\sigma_h$ = 3 cm),
toutes les 0,05 j pendant 1 j, pour une infiltration submergée ($h_{top} = 0$) dans un loam à $h_0 = -100$ cm (données de
`J06_infiltration_submergee_loam` bruitées). $\theta_r = 0{,}078$ et $\theta_s = 0{,}43$ sont connus.

Le solveur `richards_1d(par, t_out=...)` (forme mixte, Picard modifié, tridiagonal, MvG, drainage libre) est fourni : il renvoie $I$ et $h$ aux
mêmes instants (≈ 0,5 s par appel).

1. Vérifier le solveur avec les vrais paramètres (0,036 ; 1,56 ; 24,96) : RMSE par rapport aux données.
2. Écrire la fonction résidu pondéré $r_i = (y_i^{obs} - y_i(\mathbf b))/\sigma_i$ pour $\mathbf b = \log(\alpha, n, K_s)$ et lancer
   `scipy.optimize.least_squares` (bornes : $\alpha \in [0{,}005 ; 0{,}2]$, $n \in [1{,}15 ; 3]$, $K_s \in [2 ; 500]$ ; `diff_step=0.02`,
   `xtol=ftol=1e-4`) depuis (0,02 ; 1,8 ; 50). Temps de calcul, nombre d'évaluations, paramètres estimés.
3. Covariance $\mathbf C = s^2(\mathbf J^T\mathbf J)^{-1}$ avec $s^2 = 2\Phi_{min}/(N - p)$ (`res.jac`, `res.cost`) ; écarts-types, IC 95 %
   (en log puis relatifs), matrice de corrélation ; tracer les résidus.
4. Refaire l'estimation avec $I(t)$ seule : que deviennent les incertitudes et les corrélations ? Conclure sur l'identifiabilité.
5. Décrire la marche à suivre équivalente dans HYDRUS-1D (Main Processes → Inverse Solution ; types de données 0 et 1).

In [ ]:
def richards_1d(par, L=100.0, dz=1.0, h_init=-100.0, h_top=0.0, t_out=None, z_obs=(10, 30, 50),
                dt0=1e-4, dt_max=0.02, tol_th=2e-4, tol_h=1.0, max_iter=12):
    """Richards 1D en forme mixte (Celia et al. 1990), itération de Picard modifiée, système tridiagonal, Mualem-van Genuchten.
    z positif vers le haut (0 = surface). Haut : charge imposée h_top (submersion) ; bas : drainage libre (gradient unitaire).
    par = (thr, ths, alpha, n, Ks) en cm et jours. Retourne t_out, I(t_out) (infiltration cumulée, cm) et h(t_out, z_obs)."""
    thr, ths, a, n, Ks = par; m = 1 - 1 / n
    Se_f = lambda h: np.where(h < 0, (1 + (a * np.abs(h)) ** n) ** (-m), 1.0)
    theta = lambda h: thr + (ths - thr) * Se_f(h)
    K = lambda h: Ks * Se_f(h) ** 0.5 * (1 - (1 - Se_f(h) ** (1 / m)) ** m) ** 2
    C = lambda h: np.where(h < 0, (ths - thr) * a * n * m * (a * np.abs(h)) ** (n - 1) * Se_f(h) ** (1 + 1 / m), 0.0)
    N = int(round(L / dz)) + 1
    h = np.full(N, h_init, float); h[0] = h_top
    iobs = [int(round(zo / dz)) for zo in z_obs]
    t_out = np.asarray(t_out, float)
    t, dt, I, k_out = 0.0, dt0, 0.0, 0
    I_out, h_out = [], []
    while k_out < len(t_out):
        dt = min(dt, t_out[k_out] - t)
        th_old = theta(h); hk = h.copy()
        for it in range(max_iter):                                   # itérations de Picard
            Kn = K(hk); Km = 0.5 * (Kn[:-1] + Kn[1:]); thk = theta(hk)   # K inter-nodale (moyenne arithmétique)
            q = Km * ((hk[:-1] - hk[1:]) / dz + 1)                     # flux vers le bas entre les noeuds i et i+1
            rhs = np.empty(N); rhs[0] = 0.0                            # résidu : -(theta^k - theta^n)/dt + div(q)
            rhs[1:-1] = -(thk[1:-1] - th_old[1:-1]) / dt + (q[:-1] - q[1:]) / dz
            rhs[-1] = -(thk[-1] - th_old[-1]) / dt + (q[-1] - Kn[-1]) / dz     # drainage libre : q_bas = K(h_N)
            main = C(hk) / dt; main[1:-1] += (Km[:-1] + Km[1:]) / dz ** 2; main[-1] += Km[-1] / dz ** 2; main[0] = 1.0
            ab = np.zeros((3, N)); ab[1] = main; ab[0, 2:] = -Km[1:] / dz ** 2; ab[2, :-1] = -Km / dz ** 2; ab[2, 0] = 0.0
            dh = linalg.solve_banded((1, 1), ab, rhs); hk = hk + dh
            dth = np.abs(theta(hk) - thk); sat = hk >= 0                # critères de HYDRUS : theta (non saturé), h (saturé)
            if np.max(np.where(sat, 0.0, dth)) < tol_th and np.max(np.where(sat, np.abs(dh), 0.0)) < tol_h: break
        else:
            dt *= 0.3; continue                                       # non-convergence : on recommence avec un pas réduit
        Kn = K(hk)                                                    # flux en surface = flux 0-1 + stockage du demi-noeud 0
        I += (0.5 * (Kn[0] + Kn[1]) * ((hk[0] - hk[1]) / dz + 1) + (theta(hk)[0] - th_old[0]) * dz / (2 * dt)) * dt
        t += dt; h = hk
        if abs(t - t_out[k_out]) < 1e-10:
            I_out.append(I); h_out.append(h[iobs].copy()); k_out += 1
        dt = min(dt * (1.3 if it <= 3 else 0.5 if it >= 7 else 1.0), dt_max)   # contrôle du pas par le nombre d'itérations
    return t_out, np.array(I_out), np.array(h_out)


don = pd.read_csv("data/J10_inverse_donnees.csv")
t_obs = don["t_j"].to_numpy(); I_obs = don["I_cm"].to_numpy(); H_obs = don[["h10_cm", "h30_cm", "h50_cm"]].to_numpy()
sI, sH = 0.2, 3.0
THR, THS = 0.078, 0.43

# 1. vérification avec les vrais paramètres
# 2. résidu et least_squares
def residu(p, with_h=True):
    a, n, Ks = np.exp(p)
    # À COMPLÉTER : appeler richards_1d, concaténer (I - I_obs)/sI et ((H - H_obs)/sH).ravel()
    pass

# res = optimize.least_squares(residu, np.log([0.02, 1.8, 50.0]), bounds=(np.log([0.005, 1.15, 2.0]), np.log([0.2, 3.0, 500.0])), diff_step=0.02, xtol=1e-4, ftol=1e-4)
# 3. covariance, IC, corrélation, résidus
# 4. I(t) seule

## Exercice 5 — Bonus — surface de réponse et identifiabilité  *(≈ facultatif)*

Calculer $\Phi(\alpha, n)$ ($K_s$ = 24,96 fixé) sur une grille 6 × 6 ($\alpha \in [0{,}015 ; 0{,}07]$, $n \in [1{,}25 ; 2{,}1]$) avec et sans les
données de $h$ ; tracer $\log_{10}\Phi$ en isolignes et situer les vraies valeurs. Commenter la forme des vallées.

In [ ]:
al = np.linspace(0.015, 0.07, 6); nn = np.linspace(1.25, 2.1, 6)
# À COMPLÉTER : double boucle, Phi = sum(residu**2) avec/sans h, contourf

## Pour aller plus loin

* Ajouter $\theta_s$ aux paramètres ajustés : identifiable avec ces données ? Et $\theta_r$ ?
* Remplacer les données de $h$ par des teneurs en eau $\theta$ (sondes) : comparer les incertitudes.
* Estimer ($\lambda$, $R$, $\mu$) sur les BTC HYDRUS par `least_squares` autour d'un schéma numérique de l'ADE (Crank–Nicolson), puis avec
  un modèle mobile–immobile ($\theta_{im}$, $\alpha$) : quelles données permettent de distinguer les deux modèles ?
* Refaire l'exercice 4 dans HYDRUS-1D (Inverse Solution) et comparer FIT.OUT aux résultats Python.